# DES Y6 data products: review

A read-only look at every product of `msfm.apps.products.desy6` (`dev/notes/desy6_tfrecord_plan.md`, section 5).
It writes nothing. The checks that gate a product run inside the steps; this notebook is for looking, and for the two
questions the steps cannot settle on their own:

- the coverage threshold of the footprint (plan D9);
- the shear convention: κ_E × δ_g must be positive and κ_B × δ_g null, the decisive test for Y3.

Run it after the steps `nz`, `pixels`, `maps`, `values`, `config` and `noise`, in `torch_env` (healpy, h5py,
matplotlib, msfm from this checkout).

In [ ]:
from pathlib import Path

import h5py
import healpy as hp
import matplotlib.pyplot as plt
import numpy as np
import yaml

import msfm
from msfm.apps.products import desy6
from msfm.utils import catalog_desy6, files, filenames, lensing

REPO = Path(msfm.__file__).resolve().parents[1]
CONFIG = REPO / "configs/desy6/v1/default.yaml"
conf = files.load_config(str(CONFIG))

n_side = conf["analysis"]["n_side"]
n_pix = hp.nside2npix(n_side)
pix_area = hp.nside2pixarea(n_side, degrees=True)

# every product holds all survey bins; the config's z_bins select from them
survey_labels = {"metacal": [f"metacal{i}" for i in range(1, 5)], "maglim": [f"maglim{i}" for i in range(1, 7)]}
used = {sample: conf["survey"][sample]["z_bins"] for sample in survey_labels}
print(used)

## Provenance

Which steps have run, on which commit, and whether every recorded output is still the file on disk.

In [ ]:
with open(REPO / conf["files"]["provenance"]) as f:
    prov = yaml.safe_load(f)["steps"]

for step, record in prov.items():
    print(f"{step:8s} {record['date']}  msfm {record['msfm_commit'][:10]}")
    for out in record["outputs"]:
        path = REPO / out["path"]
        state = "missing" if not path.exists() else ("ok" if desy6._sha256(path) == out["sha256"] else "CHANGED")
        print(f"    {state:8s} {out['path']}")

print("not run:", [step for step in desy6.STEP_ORDER if step not in prov])

## n(z)

The fiducial n(z) of the 3x2pt FITS. Dotted lines are survey bins the config does not use. The grid starts at
z = 0.00, unlike Y3's 0.005 (plan 5.5, open point).

In [ ]:
nz_dir = str(REPO / conf["dirs"]["redshift_distributions"])
mean_z, nz_at_grid_start = {}, {}

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5), sharex=True)
for ax, (sample, labels) in zip(axes, survey_labels.items()):
    for i_bin, label in enumerate(labels, start=1):
        z, nz = np.loadtxt(filenames.get_filename_z_distribution(nz_dir, sample, i_bin, prefix=desy6.FILE_PREFIX)).T
        mean_z[label], nz_at_grid_start[label] = np.sum(z * nz) / np.sum(nz), nz[0]
        style = "-" if label in used[sample] else ":"
        ax.plot(z, nz, style, label=f"{label}, <z> = {mean_z[label]:.3f}")
    ax.set(xlabel="z", ylabel="n(z)", title=sample, xlim=(0, 2.5))
    ax.legend(fontsize=8)

print(f"grid starts at z = {z[0]}; n(z) there: {nz_at_grid_start}")

## Footprint and the coverage threshold (D9)

The pixel file stores the rotated coverage of every nside-512 pixel (`footprint/coverage`), so the threshold can be
re-derived here without healsparse. Y3 used 0.72. With more small-scale structure in the Y6 mask, a high threshold
starts to punch holes into the interior: the pixels below the threshold whose eight neighbours are all in the
footprint count those holes.

In [ ]:
with h5py.File(REPO / conf["files"]["pixels"], "r") as f:
    attrs = dict(f.attrs)
    coverage = f["footprint/coverage"][:].astype(np.float64)
    patches = f["footprint/patches"][:]
    data_vec = f["data_vec"][:]
for key, value in attrs.items():
    print(f"{key}: {value}")

threshold = attrs["coverage_threshold"]
footprint = patches[0]
in_footprint = np.zeros(n_pix, dtype=bool)
in_footprint[footprint] = True
print(
    f"\nfootprint: {footprint.size} pixels, {footprint.size * pix_area:.1f} deg^2 (mask {attrs['area_deg2']:.3f}), "
    f"mean coverage {coverage[footprint].mean():.4f}; data vector {data_vec.size} pixels"
)

# holes: below the threshold, but surrounded by the footprint
neighbours = hp.get_all_neighbours(n_side, np.arange(n_pix))
surrounded = np.all(in_footprint[neighbours] | (neighbours < 0), axis=0)
holes = surrounded & ~in_footprint
print(f"{holes.sum()} holes; their coverage: {np.round(np.sort(coverage[holes]), 3)[:30]}")

In [ ]:
touched = coverage[coverage > 0]
thresholds = np.linspace(0.05, 1, 39)

fig, (ax0, ax1, ax2) = plt.subplots(1, 3, figsize=(16, 3.5))
ax0.hist(touched, bins=100, log=True)
ax0.set(xlabel="coverage", ylabel="pixels", title=f"{touched.size} pixels touch the mask")

ax1.plot(thresholds, [np.sum(coverage >= t) * pix_area for t in thresholds])
ax1.axhline(attrs["area_deg2"], color="gray", ls=":", label="mask area")
ax1.set(xlabel="threshold", ylabel="footprint area [deg$^2$]")
ax1.legend()

n_holes = [np.sum(np.all((coverage >= t)[neighbours] | (neighbours < 0), axis=0) & (coverage < t)) for t in thresholds]
ax2.plot(thresholds, n_holes)
ax2.set(xlabel="threshold", ylabel="holes", yscale="symlog")

for ax in (ax0, ax1, ax2):
    ax.axvline(threshold, color="k", ls="--", lw=1)

In [ ]:
patch_map = np.full(n_pix, hp.UNSEEN)
for i, patch in enumerate(patches):
    patch_map[patch] = i
hp.mollview(patch_map, title="the four patches, 0 is the rotated footprint", cbar=False)
hp.graticule()

# centre of the rotated footprint, for the zooms below
lon, lat = hp.vec2ang(np.mean(hp.pix2vec(n_side, footprint), axis=1), lonlat=True)
centre = (float(np.ravel(lon)[0]), float(np.ravel(lat)[0]))

hp.orthview(np.where(coverage > 0, coverage, hp.UNSEEN), rot=centre, half_sky=True, title="coverage")
padding = np.full(n_pix, hp.UNSEEN)
padding[hp.nest2ring(n_side, data_vec)] = 0
padding[footprint] = 1
hp.orthview(padding, rot=centre, half_sky=True, title="footprint (1) and data vector padding (0)", cbar=False)

## Survey values and the config

The values file next to the published numbers (KP Table 2 for R, the methods paper for m, DES's notebook for the
area), then the step `config --check`, which writes nothing.

In [ ]:
with open(REPO / conf["files"]["survey_values"]) as f:
    values = yaml.safe_load(f)["survey"]
shear = values["metacal"]["shear_bias"]["multiplicative"]

print(f"Aeff = {values['Aeff']:.4f} deg^2 (DES {catalog_desy6.AREA_DES})\n")
print(f"{'bin':9s} {'n_gal [arcmin^-2]':>18s} {'R':>8s} {'R (KP)':>7s} {'m mu':>9s} {'m (DES)':>8s} {'m sigma':>8s}")
for i, label in enumerate(survey_labels["metacal"]):
    print(
        f"{label:9s} {values['metacal']['n_gal'][label] / 60**2:18.4f} {values['metacal']['response'][label]:8.4f} "
        f"{catalog_desy6.RESPONSE_KP[i]:7.3f} {shear['mu'][label]:9.5f} {catalog_desy6.M_MEAN_DES[i]:8.5f} "
        f"{shear['sigma'][label]:8.5f}"
    )
for label in survey_labels["maglim"]:
    note = "" if label in used["maglim"] else "  (not used)"
    print(f"{label:9s} {values['maglim']['n_gal'][label] / 60**2:18.4f}{note}")

try:
    desy6.main(["config", "--config", str(CONFIG), "--check"])
except SystemExit as error:
    print(f"config --check: STALE (exit {error.code}), run the step config")

## Maps

The cached maps of the step `maps`, all survey bins. The mean shear is not subtracted (plan D8); it is printed here
next to the per-pixel rms. For Y3 it was about 1.5% of the rms.

In [ ]:
cache = REPO / conf["dirs"]["cache"]
gamma = np.load(cache / desy6.MAPS_FILES["metacal_gamma"])  # (n_pix, 4, 2)
source_count = np.load(cache / desy6.MAPS_FILES["metacal_count"])  # (n_pix, 4)
lens_count = np.load(cache / desy6.MAPS_FILES["maglim_count"])  # (n_pix, 6)
lens_weighted = np.load(cache / desy6.MAPS_FILES["maglim_weighted_count"])  # (n_pix, 6)

for i, label in enumerate(survey_labels["metacal"]):
    n = source_count[:, i]
    g = gamma[footprint, i]
    mean = np.average(g, weights=n[footprint], axis=0)
    rms = np.sqrt(np.average(g**2, weights=n[footprint], axis=0))
    print(
        f"{label}: {n[footprint].sum() / n.sum():.4f} of {n.sum()} galaxies in the footprint, "
        f"mean (g1, g2) = ({mean[0]:+.2e}, {mean[1]:+.2e}), rms ({rms[0]:.3f}, {rms[1]:.3f})"
    )
for j, label in enumerate(survey_labels["maglim"]):
    n, w = lens_count[:, j], lens_weighted[:, j]
    print(
        f"{label}: {n[footprint].sum() / n.sum():.4f} of {n.sum()} galaxies in the footprint, "
        f"mean weight there {w[footprint].sum() / n[footprint].sum():.4f}"
    )

In [ ]:
def show(values, title, sub, **kwargs):
    m = np.full(n_pix, hp.UNSEEN)
    m[footprint] = values[footprint]
    hp.orthview(m, rot=centre, half_sky=True, title=title, sub=sub, **kwargs)


# density relative to the footprint mean, per unit covered area
plt.figure(figsize=(16, 4))
for i, label in enumerate(survey_labels["metacal"]):
    n = source_count[:, i] / np.maximum(coverage, 1e-3)
    show(n / n[footprint].mean(), f"{label} density", (1, 4, i + 1), min=0.5, max=1.5)

plt.figure(figsize=(18, 4))
for j, label in enumerate(survey_labels["maglim"]):
    n = lens_weighted[:, j] / np.maximum(coverage, 1e-3)
    show(n / n[footprint].mean(), f"{label} weighted density", (1, 6, j + 1), min=0.5, max=1.5)

# the systematics weights: mean weight per pixel
plt.figure(figsize=(18, 4))
for j, label in enumerate(survey_labels["maglim"]):
    mean_w = lens_weighted[:, j] / np.maximum(lens_count[:, j], 1)
    show(mean_w, f"{label} mean weight", (1, 6, j + 1), min=0.8, max=1.2)

plt.figure(figsize=(16, 7))
for i, label in enumerate(survey_labels["metacal"]):
    for c in range(2):
        show(gamma[:, i, c], f"{label} g{c + 1}", (2, 4, c * 4 + i + 1), min=-0.1, max=0.1)

## Noise file

The shape catalog per source bin, from the first 2 million rows of each: the response-corrected ellipticities and
the weights that the `count` shape-noise model resamples.

In [ ]:
fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(12, 3.5))
with h5py.File(REPO / conf["files"]["noise"], "r") as f:
    for i, label in enumerate(survey_labels["metacal"]):
        cat = f[f"{label}/cat"]
        sample = cat[:2_000_000].astype(np.float64)
        e_abs, w = np.hypot(sample[:, 0], sample[:, 1]), sample[:, 2]
        rms = np.sqrt(np.average(sample[:, :2] ** 2, axis=0, weights=w))
        print(f"{label}: {cat.shape[0]} galaxies (count map {source_count[:, i].sum()}), weighted rms e/R = {rms}")
        ax0.hist(e_abs, bins=200, range=(0, 1.5), histtype="step", density=True, label=label)
        ax1.hist(w, bins=200, histtype="step", density=True, label=label)
ax0.set(xlabel="|e| / R", yscale="log")
ax1.set(xlabel="w", yscale="log")
ax0.legend()

## Shear convention: κ_E × δ_g and κ_B × δ_g

The Y3 conventions (e1 +1, e2 −1, the ψ rotation of `catalog.py`) are reused for Y6 until this test passes
(memory `project_desy3_shear_convention_validated`). κ_E and κ_B come from the footprint-masked shear maps through
`lensing.mode_removal`, as on the observation path; δ_g is the weighted MagLim count per covered area. Expected:

- κ_E × δ_g **positive** where the lens bin lies in front of the source bin. A flip of both e1 and e2 makes it
  negative; a wrong e2 parity pushes it towards zero.
- κ_B × δ_g **null**. A wrong or missing ψ rotation shows up here (−3 to −6σ per band for Y3 without it).

The E/B power alone cannot decide the overall sign. The errors are Gaussian (Knox) from the measured pseudo-spectra,
with the footprint's f_sky, so they ignore the mode coupling between neighbouring bands.

In [ ]:
hp_datapath = str(REPO / conf["files"]["healpy_data"])
_, gamma2kappa_fac, _ = lensing.get_kaiser_squires_factors(3 * n_side - 1)
f_sky = in_footprint.mean()
l_max = 3 * n_side - 1
band_edges = np.array([30, 60, 100, 160, 250, 400, 630, 1000])
ell = np.arange(l_max + 1)
n_modes = np.array([np.sum(2 * ell[(ell >= lo) & (ell < hi)] + 1) for lo, hi in zip(band_edges[:-1], band_edges[1:])])


def footprint_only(m):
    """The map inside the footprint with its mean there removed, zero outside."""
    return np.where(in_footprint, m - m[in_footprint].mean(), 0.0)


def bands(cl):
    return np.array([cl[(ell >= lo) & (ell < hi)].mean() for lo, hi in zip(band_edges[:-1], band_edges[1:])])


kappa = {}
for label in used["metacal"]:
    i_survey = survey_labels["metacal"].index(label)
    g1, g2 = (np.where(in_footprint, gamma[:, i_survey, c], 0.0) for c in range(2))
    k_E, k_B = lensing.mode_removal(g1, g2, gamma2kappa_fac, n_side, hp_datapath=hp_datapath, keep_b_mode=True)
    kappa[label] = {"E": footprint_only(k_E.astype(np.float64)), "B": footprint_only(k_B.astype(np.float64))}

delta = {}
for label in used["maglim"]:
    n = lens_weighted[:, survey_labels["maglim"].index(label)]
    n_bar = n[in_footprint].sum() / coverage[in_footprint].sum()
    delta[label] = footprint_only(n / (n_bar * np.maximum(coverage, 1e-3)) - 1)

# band-averaged pseudo-spectra; the Knox error of a cross spectrum needs both autos
auto_kappa = {(label, mode): bands(hp.anafast(k[mode], lmax=l_max)) for label, k in kappa.items() for mode in "EB"}
auto_delta = {label: bands(hp.anafast(d, lmax=l_max)) for label, d in delta.items()}
cross = {}
for source, k in kappa.items():
    for lens, d in delta.items():
        for mode in "EB":
            cl = bands(hp.anafast(k[mode], d, lmax=l_max))
            sigma = np.sqrt((auto_kappa[source, mode] * auto_delta[lens] + cl**2) / (f_sky * n_modes))
            cross[source, lens, mode] = (cl, sigma)
print(f"f_sky = {f_sky:.4f}, {len(cross)} cross spectra in {len(n_modes)} bands")

In [ ]:
band_centres = 0.5 * (band_edges[:-1] + band_edges[1:])
sources, lenses = list(kappa), list(delta)


def combined(cl, sigma):
    """Inverse-variance weighted mean amplitude over the bands, in units of its error."""
    return np.sum(cl / sigma**2) / np.sqrt(np.sum(1 / sigma**2))


print(f"{'pair':20s} {'E S/N':>7s} {'B S/N':>7s} {'B chi2/dof':>11s}  lens in front")
fig, axes = plt.subplots(len(sources), len(lenses), figsize=(3 * len(lenses), 2.4 * len(sources)), sharex=True)
for r, source in enumerate(sources):
    for c, lens in enumerate(lenses):
        (cl_E, sigma_E), (cl_B, sigma_B) = cross[source, lens, "E"], cross[source, lens, "B"]
        chi2_B = np.sum((cl_B / sigma_B) ** 2)
        front = "yes" if mean_z[lens] < mean_z[source] else "no"
        print(
            f"{source + ' x ' + lens:20s} {combined(cl_E, sigma_E):7.1f} {combined(cl_B, sigma_B):7.1f} "
            f"{chi2_B:6.1f} / {len(n_modes)}  {front}"
        )

        ax = axes[r, c]
        ax.plot(band_centres, cl_E / sigma_E, "o-", label="E")
        ax.plot(band_centres, cl_B / sigma_B, "x:", label="B")
        ax.axhline(0, color="gray", lw=0.5)
        ax.set(xscale="log", title=f"{source} x {lens}")
        if c == 0:
            ax.set_ylabel("C / sigma")
        if r == len(sources) - 1:
            ax.set_xlabel("ell")
axes[0, 0].legend()
fig.tight_layout()